# MuJoCo live interactive simulations — unified notebook API

MuJoCo physics runs in the browser; Python uses **nonblocking** widget commands and callbacks in **both Binder and Colab**. Run the setup cell below, then the viewer cells. For a Python feedback controller, see the pendulum example at the end.

## 1. One-click setup

**Instructor:** Set `REPO_URL` to your public GitHub repo once before publishing. The same notebook works in Binder, Colab, or locally from the project directory.

In [ ]:
import os, sys, subprocess
from pathlib import Path
from urllib.parse import urlparse

# CHANGE ONCE: public repository containing this notebook, mujoco_widget.py and dist/*.js.
REPO_URL = "https://github.com/szafird/mujoco-example.git"

parsed = urlparse(REPO_URL)
parts = parsed.path.strip('/').removesuffix('.git').split('/')
if parsed.netloc != 'github.com' or len(parts) != 2 or 'YOUR_' in REPO_URL:
    raise ValueError('Instructor: configure REPO_URL for your public GitHub repository')
GITHUB_USER, GITHUB_REPO = parts
ASSET_BASE = f"https://{GITHUB_USER}.github.io/{GITHUB_REPO}/mujoco/"

if 'google.colab' in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()
    project_dir = Path('/content/mujoco_notebook_source')
    if not project_dir.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(project_dir)], check=True)
    else:
        subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project_dir/'requirements.txt')], check=True)
    sys.path.insert(0, str(project_dir))
else:
    project_dir = Path.cwd()
    # Binder should install requirements.txt automatically; this is safe locally too.
    try:
        import anywidget
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project_dir/'requirements.txt')], check=True)

from IPython.display import display
from mujoco_widget import MuJoCoViewer
print('Using project:', project_dir)
print('Asset base:', ASSET_BASE)

## 2. Live interactive scene

Run the cell and wait for **Ready**. The play/pause and reset buttons are in the viewer. Drag to orbit and scroll to zoom.

In [ ]:
scene_xml = """
<mujoco model="Browser demo">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 4"/>
    <geom type="plane" size="3 3 0.1" rgba="0.8 0.85 0.9 1"/>
    <body pos="0 0 1.6">
      <freejoint/>
      <geom type="sphere" size="0.18" mass="1" rgba="0.95 0.3 0.2 1"/>
    </body>
    <body pos="0.6 0 1.2">
      <freejoint/>
      <geom type="box" size="0.2 0.2 0.2" mass="1" rgba="0.2 0.55 0.85 1"/>
    </body>
  </worldbody>
</mujoco>
"""
viewer = MuJoCoViewer(scene_xml, asset_base=ASSET_BASE)
display(viewer)

## 3. Notebook commands — same in Binder and Colab

Requests return immediately. Read the result in a **later cell** (after the previous cell has finished); do not `await` or `sleep` waiting for messages.

In [ ]:
viewer.pause()
viewer.request_sim_state()  # nonblocking; a reply updates latest_sim_state

In [ ]:
state = viewer.latest_sim_state
print('Most recent state:', state)
if state:
    print('t:', state['time'], 'qpos:', state['qpos'])

In [ ]:
viewer.reset()
viewer.set_gravity(-3.0)
viewer.step(25)  # advances 25 * 0.005 = 0.125 s while paused

In [ ]:
print('After stepping:', viewer.latest_sim_state)
# Return to live physics when you want:
# viewer.play()

## 4. Python PD control of a motorized pendulum

The controller is **ordinary Python**, but it is called by widget callbacks between browser physics steps. Crucially, the notebook cell starts the run and returns immediately, so Colab can deliver subsequent callbacks. This is suitable for teaching demonstrations, not high-frequency or hard real-time control.

In [ ]:
pendulum_xml = """
<mujoco model="Motorized pendulum">
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 4"/>
    <geom type="plane" size="2 2 0.1" rgba="0.85 0.9 0.94 1"/>
    <body name="pendulum" pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0" pos="0 0 0" ref="0.8"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.8" size="0.055" mass="0.4" rgba="0.2 0.55 0.9 1"/>
    </body>
  </worldbody>
  <actuator>
    <motor name="hinge_torque" joint="hinge" gear="1"/>
  </actuator>
</mujoco>
"""
pendulum = MuJoCoViewer(pendulum_xml, asset_base=ASSET_BASE)
display(pendulum)

**Start after pendulum displays Ready.** `run_controller` resets *neither* position nor time; it pauses the viewer and starts from the current state. The control goal is to stabilize the hanging position at angle zero.

In [ ]:
def pd_control(state):
    angle = state['qpos'][0]
    angular_velocity = state['qvel'][0]
    kp, kd = 8.0, 1.5
    return -kp * angle - kd * angular_velocity

run = pendulum.run_controller(pd_control, steps=150, substeps=5)
print('Started:', run)  # returns immediately; no await/while/sleep

In [ ]:
print('Controller progress:', run)
print('Latest state:', pendulum.latest_sim_state)
# If needed: run.cancel()

## 5. Inspect asynchronous replies with a Python callback (optional)

In [ ]:
def show_snapshot(state):
    print(f"State snapshot arrived: t={state['time']:.3f}, qpos={state['qpos']}")

viewer.on_sim_state(show_snapshot)
viewer.request_sim_state()
# The callback prints after this cell is allowed to finish.
# To deregister later: viewer.on_sim_state(show_snapshot, remove=True)